# Understanding the Difference between Embedding Layers and Linear Layers

Embedding layers in PyTorch accomplish the same as linear layers that perform matrix multiplications. The reason we use embedding layers is computational efficiency.

In [1]:
import torch

## Using nn.Embedding

In [2]:
# Token IDs in an LLM context
idx = torch.tensor([2, 3, 1])

# To get number of rows we will obtain the largest token ID + 1
# This is the same as vocabulary size
num_idx = max(idx) + 1

# Desired embedding dimension is a hyperparameter
out_dim = 5

Simple embedding layer.

In [3]:
torch.manual_seed(123)
embedding = torch.nn.Embedding(num_idx, out_dim)

We can peek at the weights of the embedding layer:

In [4]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

We use embedding layers to obtain the vector representation of a training example with ID 1 (or any other ID):

In [5]:
embedding(torch.tensor([1]))

tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

In [6]:
embedding(torch.tensor([2]))

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315]],
       grad_fn=<EmbeddingBackward0>)

Let's convert all the training examples we have defined previously:

In [7]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

## Using nn.Linear
We will demonstrate that the embedding layer above accomplishes exactly the same as an `nn.Linear` layer on a one-hot encoded representation in PyTorch.

Let's convert token IDs into a one-hot representation:

In [8]:
idx

tensor([2, 3, 1])

In [9]:
onehot = torch.nn.functional.one_hot(idx)
onehot

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

Initialising a `Linear` layer that carries out a matrix multiplication $XW^T$:

In [10]:
torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)
linear.weight

Parameter containing:
tensor([[-0.2039,  0.0166, -0.2483,  0.1886],
        [-0.4260,  0.3665, -0.3634, -0.3975],
        [-0.3159,  0.2264, -0.1847,  0.1871],
        [-0.4244, -0.3034, -0.1836, -0.0983],
        [-0.3814,  0.3274, -0.1179,  0.1605]], requires_grad=True)

**Note:** Linear layer in PyTorch is also initialised with small random weights. To directly compare it to the `Embedding` layer above we need to reassign them here:

In [11]:
linear.weight = torch.nn.Parameter(embedding.weight.T)

Now we can use the linear layer on the one-hot encoded representation of the inputs:

In [12]:
linear(onehot.float())

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

In [13]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- Since all but one index in each one-hot encoded row are 0 (by design), this matrix multiplication is essentially the same as a lookup of the one-hot elements
- This use of matrix multiplication on one-hot encodings is equivalent to the embedding layer lookup but can be inefficient if we work with large embedding matrices, because there are a lot of wasteful multiplications by zero